In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import astropy.constants as c
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline, RectBivariateSpline
from matplotlib import cm, colors
from matplotlib.ticker import LogLocator, FuncFormatter
import niceplots.utils as nicepl

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

nicepl.initPlot()
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
Cp

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.LIMsurvey import covariance as scov
from SSLimPy.LIMsurvey import power_spectrum as spobs

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
    "Smooth_resolution": True,
    # "nonlinearMatpow": False,
}

h = 0.677
cosmodict={
    "h": h,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
    "nonlinear_bias": "HMF",
}

# MeerKlass HI survey L-Band

In [ ]:
surveyspecs_MK = {
    "Tsys_NEFD": 23 * u.K,
    "Nfeeds": 2,
    "nD": 64,
    "beam_FWHM": 0.3 * u.deg,
    "nu": 1420 * u.MHz,
    "nuObs": 997.4 * u.MHz,
    "Delta_nu": 52.4 * u.MHz,
    "dnu": 0.209 * u.MHz,
    "tobs": 62 * u.h,
    "Omega_field": 236 * u.deg**2
}

astrodict_HI={
    "model_type": "ML",
    "model_name": "L_from_MHI_VN",
    "model_par": {
        "alpha": 0.53,
        "M0": 1.5e10 * u.Msun / h,
        "Mmin":6.0e11 * u.Msun / h,
        "do_quench": False,
    },
    "sigma_scatter" : 0.0,
}

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_HI,
    obspars_dict=surveyspecs_MK,
)

In [ ]:
myastro_HI = myssl.current_astro
pobs_HI = spobs.PowerSpectra(myastro_HI, settings= {
    "kmin": 4.e-2 * u.Mpc**-1,
    "kmax": 1.3 * u.Mpc**-1,
    "nk" : 50,
})

In [ ]:
pobs_HI.survey_specs.get_redshifts()

In [ ]:
pobs_HI.survey_specs.sigma_Noise().to(u.mK)

In [ ]:
G_SS_cov_HI = scov.Covariance(pobs_HI).gaussian_nonoise_cov()[:, 0, 0, 0]
G_SN_cov_HI = scov.Covariance(pobs_HI).gaussian_signalxnoise_cov()[:, 0, 0, 0]
G_NN_cov_HI = scov.Covariance(pobs_HI).gaussian_noise_cov()[:, 0, 0, 0]
G_cov_HI = G_SS_cov_HI + G_SN_cov_HI + G_NN_cov_HI

plt.errorbar(pobs_HI.k, pobs_HI.Pk_0bs.squeeze(), np.sqrt(G_cov_HI))
plt.loglog()
plt.ylim(2e4, 2e9)

In [ ]:
NG_cov_HI = scov.nonGuassianCov(pobs_HI).compute_nG_Cov().squeeze()
SSC_cov_HI = scov.SuperSampleCovariance(pobs_HI).compute_SSC().squeeze()

In [ ]:
cov_tot_HI = np.diag(G_cov_HI) + NG_cov_HI + SSC_cov_HI
dcov_tot_HI = np.diag(cov_tot_HI)
corr_HI = cov_tot_HI / np.sqrt(np.outer(dcov_tot_HI, dcov_tot_HI))

In [ ]:
fig, ax = plt.subplots()

plt.imshow(corr_HI, vmax=1, origin="lower")
plt.colorbar()

# ---------- k-axis ticks ----------
k = pobs_HI.k
N = len(k)

# choose ~6 ticks across the matrix
nticks = 6
tick_pos = np.linspace(0, N-1, nticks, dtype=int)

ax.set_xticks(tick_pos)
ax.set_yticks(tick_pos)

ax.set_xticklabels([f"{k.value[i]:.2e}" for i in tick_pos], rotation=45)
ax.set_yticklabels([f"{k.value[i]:.2e}" for i in tick_pos])

ax.set_xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
ax.set_ylabel(r"$k\ [{\rm Mpc}^{-1}]$")

# ---------- title ----------
ax.set_title("HI, MeerKlass-like Setup")
plt.savefig("output/HI_Corr.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

CI   = np.diag(G_SS_cov_HI).to((pobs_HI.Pk_0bs.unit)**2)
CII  = np.diag(G_SN_cov_HI).to((pobs_HI.Pk_0bs.unit)**2)
CIII = np.diag(G_NN_cov_HI).to((pobs_HI.Pk_0bs.unit)**2)
CNG  = NG_cov_HI.to((pobs_HI.Pk_0bs.unit)**2)
CSSC = SSC_cov_HI.to((pobs_HI.Pk_0bs.unit)**2)

covs = [CI, CII, CIII, CNG, CSSC]

labels = [
    r"$\mathrm{Cov}^{\rm TT}_{\rm G}$",
    r"$\mathrm{Cov}^{\rm TN}_{\rm G}$",
    r"$\mathrm{Cov}^{\rm NN}_{\rm G}$",
    r"$\mathrm{Cov}_{\rm NG}$",
    r"$\mathrm{Cov}_{\rm SSC}$"
]

# Stack + determine dominant contribution
cov_stack = np.stack(np.abs(covs), axis=0)
largest = np.argmax(np.abs(cov_stack), axis=0)

# Colors
Cs = sns.color_palette("colorblind", n_colors=5)
cmap = ListedColormap(Cs)

fig, ax = plt.subplots()

im = ax.imshow(largest, cmap=cmap, origin='lower', vmin=0, vmax=5)

# ---------- k-axis ticks ----------
k = pobs_HI.k
N = len(k)

# choose ~6 ticks across the matrix
nticks = 6
tick_pos = np.linspace(0, N-1, nticks, dtype=int)

ax.set_xticks(tick_pos)
ax.set_yticks(tick_pos)

ax.set_xticklabels([f"{k.value[i]:.2e}" for i in tick_pos], rotation=45)
ax.set_yticklabels([f"{k.value[i]:.2e}" for i in tick_pos])

ax.set_xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
ax.set_ylabel(r"$k\ [{\rm Mpc}^{-1}]$")

# ---------- title ----------
ax.set_title("HI, MeerKlass-like Setup")

# ---------- legend ----------
handles = [
    plt.matplotlib.patches.Patch(color=Cs[i], label=labels[i])
    for i in range(len(labels))
]

ax.legend(
    handles=handles,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.savefig("output/HI_Cov.png")

In [ ]:
Cs = seaborn.color_palette("colorblind")
color = iter(Cs)
for cov, label in zip(covs, labels):
    c = next(color)
    plt.loglog(k, np.diag(cov), c=c, label=label)
    plt.loglog(k, -np.diag(cov), c=c, ls="--")
plt.legend()
plt.xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}(k, k)\ [\mu\mathrm{K}^4\,\mathrm{Mpc}^6]$")
plt.title(r"$\mathrm{HI}$, MeerKlass-like Setup")
plt.savefig("output/HI_digaonal.png")

# MeerKlass UHF Band

In [ ]:
surveyspecs_MK = {
    "Tsys_NEFD": 26 * u.K,
    "Nfeeds": 2,
    "nD": 64,
    "beam_FWHM": 0.3 * u.deg,
    "nu": 1420 * u.MHz,
    "nuObs": 745 * u.MHz,
    "Delta_nu": 185 * u.MHz,
    "dnu": 0.209 * u.MHz,
    "tobs": 60 * u.h,
    "Omega_field": 250 * u.deg**2
}

astrodict_HI={
    "model_type": "ML",
    "model_name": "L_from_MHI_VN",
    "model_par": {
        "alpha": 0.53,
        "M0": 1.5e10 * u.Msun / h,
        "Mmin":6.0e11 * u.Msun / h,
        "do_quench": False,
    },
    "sigma_scatter" : 0.0,
}

In [ ]:
pobs_HI = myssl.compute(
    myssl.current_cosmology.cosmopars,
    myssl.current_halomodel.haloparams,
    astrodict_HI,
    surveyspecs_MK,
    pobs_settings={
        "kmin":3e-2 * u.Mpc**-1,
        "kmax":0.4 * u.Mpc**-1,
        "nk":50,
    },
    output=["Power spectrum"]
)["Power spectrum"]
myastro_HI = pobs_HI.astro

In [ ]:
pobs_HI.survey_specs.sigma_Noise().to(u.mK)

In [ ]:
pobs_HI.z

In [ ]:
G_SS_cov_HI = scov.Covariance(pobs_HI).gaussian_nonoise_cov()[:, 0, 0, 0]
G_SN_cov_HI = scov.Covariance(pobs_HI).gaussian_signalxnoise_cov()[:, 0, 0, 0]
G_NN_cov_HI = scov.Covariance(pobs_HI).gaussian_noise_cov()[:, 0, 0, 0]
G_cov_HI = G_SS_cov_HI + G_SN_cov_HI + G_NN_cov_HI

plt.errorbar(pobs_HI.k, pobs_HI.Pk_0bs.squeeze(), np.sqrt(G_cov_HI))
plt.loglog()

In [ ]:
NG_cov_HI = scov.nonGuassianCov(pobs_HI).compute_nG_Cov().squeeze()
SSC_cov_HI = scov.SuperSampleCovariance(pobs_HI).compute_SSC().squeeze()

In [ ]:
cov_tot_HI = np.diag(G_cov_HI) + NG_cov_HI + SSC_cov_HI
dcov_tot_HI = np.diag(cov_tot_HI)
corr_HI = cov_tot_HI / np.sqrt(np.outer(dcov_tot_HI, dcov_tot_HI))

In [ ]:
fig, ax = plt.subplots()

plt.imshow(corr_HI, vmax=1, origin="lower")
plt.colorbar()

# ---------- k-axis ticks ----------
k = pobs_HI.k
N = len(k)

# choose ~6 ticks across the matrix
nticks = 6
tick_pos = np.linspace(0, N-1, nticks, dtype=int)

ax.set_xticks(tick_pos)
ax.set_yticks(tick_pos)

ax.set_xticklabels([f"{k.value[i]:.2e}" for i in tick_pos], rotation=45)
ax.set_yticklabels([f"{k.value[i]:.2e}" for i in tick_pos])

ax.set_xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
ax.set_ylabel(r"$k\ [{\rm Mpc}^{-1}]$")

# ---------- title ----------
ax.set_title("HI, MeerKlass-like UHF Setup")
plt.savefig("output/HI_UHF_Corr.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

CI   = np.diag(G_SS_cov_HI).to((pobs_HI.Pk_0bs.unit)**2)
CII  = np.diag(G_SN_cov_HI).to((pobs_HI.Pk_0bs.unit)**2)
CIII = np.diag(G_NN_cov_HI).to((pobs_HI.Pk_0bs.unit)**2)
CNG  = NG_cov_HI.to((pobs_HI.Pk_0bs.unit)**2)
CSSC = SSC_cov_HI.to((pobs_HI.Pk_0bs.unit)**2)

covs = [CI, CII, CIII, CNG, CSSC]

labels = [
    r"$\mathrm{Cov}^{\rm TT}_{\rm G}$",
    r"$\mathrm{Cov}^{\rm TN}_{\rm G}$",
    r"$\mathrm{Cov}^{\rm NN}_{\rm G}$",
    r"$\mathrm{Cov}_{\rm NG}$",
    r"$\mathrm{Cov}_{\rm SSC}$"
]

# Stack + determine dominant contribution
cov_stack = np.stack(np.abs(covs), axis=0)
largest = np.argmax(np.abs(cov_stack), axis=0)

# Colors
Cs = sns.color_palette("colorblind", n_colors=5)
cmap = ListedColormap(Cs)

fig, ax = plt.subplots()

im = ax.imshow(largest, cmap=cmap, origin='lower', vmin=0, vmax=5)

# ---------- k-axis ticks ----------
k = pobs_HI.k
N = len(k)

# choose ~6 ticks across the matrix
nticks = 6
tick_pos = np.linspace(0, N-1, nticks, dtype=int)

ax.set_xticks(tick_pos)
ax.set_yticks(tick_pos)

ax.set_xticklabels([f"{k.value[i]:.2e}" for i in tick_pos], rotation=45)
ax.set_yticklabels([f"{k.value[i]:.2e}" for i in tick_pos])

ax.set_xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
ax.set_ylabel(r"$k\ [{\rm Mpc}^{-1}]$")

# ---------- title ----------
ax.set_title("HI, MeerKlass-like UHF Setup")

# ---------- legend ----------
handles = [
    plt.matplotlib.patches.Patch(color=Cs[i], label=labels[i])
    for i in range(len(labels))
]

ax.legend(
    handles=handles,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.savefig("output/HI_UHF_Cov.png")

In [ ]:
Cs = seaborn.color_palette("colorblind")
color = iter(Cs)
for cov, label in zip(covs, labels):
    c = next(color)
    plt.loglog(k, np.diag(cov), c=c, label=label)
    plt.loglog(k, -np.diag(cov), c=c, ls="--")
plt.legend()
plt.xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}(k, k)\ [\mu\mathrm{K}^4\,\mathrm{Mpc}^6]$")
plt.title(r"$\mathrm{HI}$, MeerKlass-like UHF Setup")
plt.savefig("output/HI_UHF_digaonal.png")

# CCAT' EoR Survey

In [ ]:
nu = 280 * u.GHz
FWHMnu = nu / 100
dnu = FWHMnu / np.sqrt(8 * np.log(2))

surveyspecs_CCATp = {
    "Tsys_NEFD": 81e-1 * u.mJy * u.s**0.5, #/ np.sqrt(6912)
    "Nfeeds": 6912,
    "nD": 1,
    "beam_FWHM": 48 * u.arcsec,
    "nu": 1.901 * u.THz,
    "nuObs": nu,
    "Delta_nu": 40 * u.GHz,
    "dnu": dnu,
    "tobs": 2000 / 42 * u.h,
    "Omega_field": 4 * u.deg**2,
    "do_Jysr": True,
}

astrodict_CCATp={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
}

In [ ]:
pobs_CII = myssl.compute(
    myssl.current_cosmology.cosmopars,
    myssl.current_halomodel.haloparams,
    astrodict_CCATp,
    surveyspecs_CCATp,
    pobs_settings={
        "kmin":3e-2 * u.Mpc**-1,
        "kmax":5 * u.Mpc**-1,
        "nk":50,
    },
    output=["Power spectrum"]
)["Power spectrum"]
myastro_CII = pobs_CII.astro

In [ ]:
pobs_CII.survey_specs.detector_noise()

In [ ]:
plt.loglog(pobs_CII.k, pobs_CII.Pk_0bs.squeeze())
plt.loglog(pobs_CII.k, pobs_CII.survey_specs.detector_noise() * np.ones_like(pobs_CII.k.value))

In [ ]:
G_SS_cov_CII = scov.Covariance(pobs_CII).gaussian_nonoise_cov()[:, 0, 0, 0]
G_SN_cov_CII = scov.Covariance(pobs_CII).gaussian_signalxnoise_cov()[:, 0, 0, 0]
G_NN_cov_CII = scov.Covariance(pobs_CII).gaussian_noise_cov()[:, 0, 0, 0]
G_cov_CII = G_SS_cov_CII + G_SN_cov_CII + G_NN_cov_CII

plt.errorbar(pobs_CII.k, pobs_CII.Pk_0bs.squeeze(), np.sqrt(G_cov_CII))
plt.loglog()

In [ ]:
NG_cov_CII = scov.nonGuassianCov(pobs_CII).compute_nG_Cov().squeeze()
SSC_cov_CII = scov.SuperSampleCovariance(pobs_CII).compute_SSC().squeeze()

In [ ]:
cov_tot_CII = np.diag(G_cov_CII) + NG_cov_CII + SSC_cov_CII
dcov_tot_CII = np.diag(cov_tot_CII)
corr_CII = cov_tot_CII / np.sqrt(np.outer(dcov_tot_CII, dcov_tot_CII))

In [ ]:
fig, ax = plt.subplots()

plt.imshow(corr_CII, vmax=1, origin="lower")
plt.colorbar()

# ---------- k-axis ticks ----------
k = pobs_CII.k
N = len(k)

# choose ~6 ticks across the matrix
nticks = 6
tick_pos = np.linspace(0, N-1, nticks, dtype=int)

ax.set_xticks(tick_pos)
ax.set_yticks(tick_pos)

ax.set_xticklabels([f"{k.value[i]:.2e}" for i in tick_pos], rotation=45)
ax.set_yticklabels([f"{k.value[i]:.2e}" for i in tick_pos])

ax.set_xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
ax.set_ylabel(r"$k\ [{\rm Mpc}^{-1}]$")

# ---------- title ----------
ax.set_title(r"$\mathrm{[CII]}$, $10\times$CCAT'-like Setup")
plt.savefig("output/CII_Corr.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

CI   = np.diag(G_SS_cov_CII).to((pobs_CII.Pk_0bs.unit)**2)
CII  = np.diag(G_SN_cov_CII).to((pobs_CII.Pk_0bs.unit)**2)
CIII = np.diag(G_NN_cov_CII).to((pobs_CII.Pk_0bs.unit)**2)
CNG  = NG_cov_CII.to((pobs_CII.Pk_0bs.unit)**2)
CSSC = SSC_cov_CII.to((pobs_CII.Pk_0bs.unit)**2)

covs = [CI, CII, CIII, CNG, CSSC]

labels = [
    r"$\mathrm{Cov}^{\rm TT}_{\rm G}$",
    r"$\mathrm{Cov}^{\rm TN}_{\rm G}$",
    r"$\mathrm{Cov}^{\rm NN}_{\rm G}$",
    r"$\mathrm{Cov}_{\rm NG}$",
    r"$\mathrm{Cov}_{\rm SSC}$"
]

# Stack + determine dominant contribution
cov_stack = np.stack(np.abs(covs), axis=0)
largest = np.argmax(np.abs(cov_stack), axis=0)

# Colors
Cs = sns.color_palette("colorblind", n_colors=5)
cmap = ListedColormap(Cs)

fig, ax = plt.subplots()

im = ax.imshow(largest, cmap=cmap, origin='lower', vmin=0, vmax=5)

# ---------- k-axis ticks ----------
k = pobs_CII.k
N = len(k)

# choose ~6 ticks across the matrix
nticks = 6
tick_pos = np.linspace(0, N-1, nticks, dtype=int)

ax.set_xticks(tick_pos)
ax.set_yticks(tick_pos)

ax.set_xticklabels([f"{k.value[i]:.2e}" for i in tick_pos], rotation=45)
ax.set_yticklabels([f"{k.value[i]:.2e}" for i in tick_pos])

ax.set_xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
ax.set_ylabel(r"$k\ [{\rm Mpc}^{-1}]$")

# ---------- title ----------
ax.set_title(r"$\mathrm{[CII]}$, $10\times$CCAT'-like Setup")

# ---------- legend ----------
handles = [
    plt.matplotlib.patches.Patch(color=Cs[i], label=labels[i])
    for i in range(len(labels))
]

ax.legend(
    handles=handles,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.savefig("output/CII_Cov.png")

In [ ]:
Cs = seaborn.color_palette("colorblind")
color = iter(Cs)
for cov, label in zip(covs, labels):
    c = next(color)
    plt.loglog(k, np.diag(cov), c=c, label=label)
    plt.loglog(k, -np.diag(cov), c=c, ls="--")
plt.legend()
plt.xlabel(r"$k\ [{\rm Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}(k, k)\ [\mathrm{Jy}^4\,\mathrm{sr}^{-4}\,\mathrm{Mpc}^6]$")
plt.title(r"$\mathrm{[CII]}$, $10\times$ CCAT'-like Setup")
plt.savefig("output/CII_Diag.png")